In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from delta.tables import DeltaTable

In [0]:
product_m_desc = spark.read.format("parquet").load("abfss://bronze@intechaccstorage.dfs.core.windows.net/ProductModelProductDescription")
product_m_desc.limit(5).display()

ProductModelID,ProductDescriptionID,Culture,rowguid,ModifiedDate,_rescued_data
1,1199,en,4d00b649-027a-4f99-a380-f22a46ec8638,2007-06-01 00:00:00.0000000,null
1,1467,ar,7de7204e-4efc-40f7-b9e6-0ceb4162399c,2007-06-01 00:00:00.0000000,null
1,1589,fr,20bffcf4-bfa6-400b-bf34-9a689779eae4,2007-06-01 00:00:00.0000000,null
1,1712,th,af18bc29-f378-4512-a59b-d55217f4f82b,2007-06-01 00:00:00.0000000,null
1,1838,he,23830ac3-3b16-49d1-9ba1-3a8996f0fa7a,2007-06-01 00:00:00.0000000,null


In [0]:
product_m_desc = product_m_desc.withColumnRenamed('ProductModelID', 'Product_model_id')\
                            .withColumnRenamed('ProductDescriptionID', 'Product_desc_id')
                           
product_m_desc = product_m_desc.drop("rowguid", "ModifiedDate", "_rescued_data")

In [0]:
product_m_desc.columns

['Product_model_id', 'Product_desc_id', 'Culture']

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricksintech.silver.product_model_description
(
    Product_model_id STRING, 
    Product_desc_id STRING,
    Culture STRING
 
) 
USING DELTA
LOCATION 'abfss://silver@intechaccstorage.dfs.core.windows.net/product_model_description';

In [0]:
# Reference target Delta table
silver_table = DeltaTable.forName(spark, "databricksintech.silver.product_model_description")

# Execute the Merge (Upsert)
silver_table.alias("target").merge(
    source = product_m_desc.alias("source"),
    condition = (
        "target.Product_model_id = source.Product_model_id"
        
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Done!")

Done!
